<a href="https://colab.research.google.com/github/amber-pan/Self-taught-AI-Engineer/blob/main/MSAI_ML_incremental_Capstone_8_consolidated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Created by: Amber Pan
**Movie recommendation:**

Study the various Recommendation Techniques for recommending movies using movies.csv, ratings.csv datasets

1. Load and merge ratings.csv and movies.csv.
2. Build a User-Item matrix.
3. User-Based Collaborative Filtering and predict User 1's rating for movieId 32.
4. Item-Based Collaborative Filtering and find 10 movies similar to Jurassic Park (1993).
5. KNNBasic model evaluation.
6. SVD model evaluation.
7. NMF model evaluation.
8. Compare RMSE scores across models.

!pip install surprise

import numpy as np
import pandas as pd

from surprise import Dataset
from surprise import Reader
from surprise import KNNBasic
from surprise import SVD
from surprise import NMF
from surprise.model_selection import cross_validate

# Load movies.csv and ratings.csv dataset
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

print(ratings.shape)
print(movies.shape)

ratings.head()

movies.head()

# Merge both data frames on movieid

# Merge datasets
movie_data = pd.merge(
    ratings,
    movies,
    on="movieId",
    how="inner"
)

movie_data.head()

# create user-item matrix
# # Create User-Item Matrix (Hint: Use pandas pivot_table method with index = 'userId', columns = 'title', values = 'rating' )
user_item_matrix = movie_data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

print(user_item_matrix.shape)

user_item_matrix.head()

# fill missing values with user mean
# Perform User-based Collaborative Filtering
# Fill the row-wise NaNs in the User-Item Matrix with the corresponding user's mean ratings, and find the Pearson correlation between users

user_mean_matrix = user_item_matrix.apply(
    lambda row: row.fillna(row.mean()),
    axis=1
)

user_mean_matrix.head()

# use correlation matrix
# Choose the correlation of all users with only User 1
# Sort the User 1 correlation in the descending order

user_corr = user_mean_matrix.T.corr(method='pearson')

user_corr.head()

# use user 1 correlation
#. Drop the NaN values generated in the correlation matrix

user1_corr = user_corr[1]

user1_corr = user1_corr.sort_values(
    ascending=False
)

user1_corr = user1_corr.dropna()

user1_corr.head(10)

# top 50 similar users
# Choose the top 50 users that are highly correlated to User 1

top50_users = user1_corr.iloc[1:51]

top50_users.head()

# Obtain movie 32 title info
#
movie32_title = movies.loc[
    movies['movieId'] == 32,
    'title'
].values[0]

print(movie32_title)

# gather rating for similar users
#Predict the rating that User 1 might give for the movie with movieid 32 based on the top 50 user correlation matrix
# (Hint: Predicted rating = sum of [(weights) * (ratings)] / sum of (weights ). Here, weights is the correlation of the corresponding user with the first user). That is, the predicted rating is calculated as the weighted average of k similar users
similar_users = top50_users.index

ratings_movie32 = user_item_matrix.loc[
    similar_users,
    movie32_title
]

prediction_df = pd.DataFrame({
    'weight': top50_users,
    'rating': ratings_movie32
})

prediction_df = prediction_df.dropna()

prediction_df.head()

# weighted average
predicted_rating = (
    np.sum(
        prediction_df['weight'] *
        prediction_df['rating']
    )
    /
    np.sum(prediction_df['weight'])
)

print("Predicted Rating for User 1:", predicted_rating)

# check if the user 1 has rated movie 32 in original movie_data
selection_rating = movie_data.loc[
    (movie_data['userId'] == 1) &
    (movie_data['movieId'] == 32),
    'rating'
]

if not selection_rating.empty:
    original_rating = selection_rating.values[0]
    print(f"Original Rating for User 1 for movie {movie32_title}:", original_rating)
else:
    print(f"User 1 has not rated movie {movie32_title} (movieId 32) in the dataset.")


# check what user 1 has rated movie 32 in user_item_matrix
user_item_matrix.loc[
    1,
    movie32_title
]

6. Item-Based Collaborative Filtering
Fill Missing Values with Movie Mean


# fill missing value with movie mean
# Fill the column-wise NaN's in the User-Item Matrix with the corresponding movie's mean ratings, and find Pearson correlation between movies


item_mean_matrix = user_item_matrix.copy()

item_mean_matrix = item_mean_matrix.apply(
    lambda col: col.fillna(col.mean()),
    axis=0
)

item_mean_matrix.head()

# movie correlation matrix

movie_corr = item_mean_matrix.corr(
    method='pearson'
)

movie_corr.head()

# correlation with Jurassic Park 1993
# Choose the correlation of all movies with the movie Jurassic Park (1993) only
# Drop the NaN values generated in the correlation matrix
jurassic_corr = movie_corr[
    'Jurassic Park (1993)'
]

jurassic_corr = jurassic_corr.sort_values(
    ascending=False
)

jurassic_corr = jurassic_corr.dropna()

jurassic_corr.head(20)


# top 10 similar movies
# Sort the  Jurassic Park movie correlation in descending order

# Find 10 movies similar to the movie Jurassic Park (1993)

similar_movies = jurassic_corr.iloc[1:11]

print("Top 10 Similar Movies\n")

for movie in similar_movies.index:
    print(movie)

Perform KNNBasic, SVD, NMF Model-based Collaborative Filtering




# prepare data for Surprise models
# Reader = metadata that tells Surprise how to interpret the ratings data
# Tell Surprise what a valid rating looks like.
# Initialize KNNBasic with similarity configuration as Mean Squared Distance Similarity (msd), 20 neighbors and cross-validate 5 folds against measure RMSE.
# (Hint: cross_validate(algo=algo, data=data, measures=['RMSE'], cv=5, verbose=True))

reader = Reader(
    rating_scale=(
        ratings.rating.min(),
        ratings.rating.max()
    )
)

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

# KNNBasic model with Mean Square Distance
sim_options = {
    'name': 'msd',
    'user_based': True
}

knn_model = KNNBasic(
    k=20,
    sim_options=sim_options
)

#cross validation
knn_results = cross_validate(
    algo=knn_model,
    data=data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

print("KNN RMSE Mean:",
      np.mean(knn_results['test_rmse']))
# The cross_validate function, which was used, automatically performs cross-validation. This means it splits the data into multiple folds (in this case, 5 folds as indicated by cv=5). For each fold, it uses a portion of the data for training and the remaining portion as the test set. The test_rmse values that you see are the Root Mean Squared Error calculated on the test set for each of these 5 folds. Taking the np.mean() of these values gives you the average RMSE across all the test sets, providing a more robust estimate of the model's performance.

# SVD model
# Initialize Singular Value Decomposition (SVD) and  cross-validate 5 folds against measure RMSE.

svd_model = SVD()

svd_results = cross_validate(
    algo=svd_model,
    data=data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

print("SVD RMSE Mean:",
      np.mean(svd_results['test_rmse']))

# NMF model
#NMF discovers hidden movie characteristics and hidden user preferences. It represents both users and movies using only positive latent factors, then predicts ratings by matching user preferences with movie characteristics.
# Initialize Non-Negative Matrix Factorization (NMF) and cross-validate 5 folds against measure RMSE.

nmf_model = NMF()

nmf_results = cross_validate(
    algo=nmf_model,
    data=data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

print("NMF RMSE Mean:",
      np.mean(nmf_results['test_rmse']))

# compare model performances
# Print best score and best params from Cross Validate on all the models built.
results = pd.DataFrame({
    'Model': ['KNNBasic', 'SVD', 'NMF'],
    'RMSE': [
        np.mean(knn_results['test_rmse']),
        np.mean(svd_results['test_rmse']),
        np.mean(nmf_results['test_rmse'])
    ]
})

results = results.sort_values(
    by='RMSE'
)

results

	Model	RMSE
1	SVD	0.873294
2	NMF	0.923847
0	KNNBasic	0.939502


#best model
best_model = results.iloc[0]

print(
    f"Best Model: {best_model['Model']}"
)

print(
    f"Best RMSE: {best_model['RMSE']:.4f}"
)

### **use grid search so that you can get best params**
only grid search can provide params so

#  obrain best score and best params from Grid Search Cross Validate on all the models built.
from surprise.model_selection import GridSearchCV

param_grid_knn = {
    'k': [10, 20, 30, 40],
    'sim_options': {
        'name': ['msd', 'cosine', 'pearson'],
        'user_based': [True, False]
    }
}

gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse'],
    cv=5,
    n_jobs=-1
)

gs_knn.fit(data)

print("KNN Best RMSE:")
print(gs_knn.best_score['rmse'])

print("\nKNN Best Parameters:")
print(gs_knn.best_params['rmse'])

param_grid_svd = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}

gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse'],
    cv=5,
    n_jobs=-1
)

gs_svd.fit(data)

print("SVD Best RMSE:")
print(gs_svd.best_score['rmse'])

print("\nSVD Best Parameters:")
print(gs_svd.best_params['rmse'])

param_grid_nmf = {
    'n_factors': [15, 30, 50],
    'n_epochs': [50, 100],
    'reg_pu': [0.02, 0.06],
    'reg_qi': [0.02, 0.06]
}

gs_nmf = GridSearchCV(
    NMF,
    param_grid_nmf,
    measures=['rmse'],
    cv=5,
    n_jobs=-1
)

gs_nmf.fit(data)

print("NMF Best RMSE:")
print(gs_nmf.best_score['rmse'])

print("\nNMF Best Parameters:")
print(gs_nmf.best_params['rmse'])

results = pd.DataFrame({
    'Model': ['KNNBasic', 'SVD', 'NMF'],
    'Best RMSE': [
        gs_knn.best_score['rmse'],
        gs_svd.best_score['rmse'],
        gs_nmf.best_score['rmse']
    ]
})

results = results.sort_values(
    by='Best RMSE'
)

results

best_model = results.iloc[0]

print(f"Best Model: {best_model['Model']}")
print(f"Best RMSE: {best_model['Best RMSE']:.4f}")